# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

#### To prioritize which pages tp fix, the pages will be ranked by combining the four weighted factors
##### Visibility (40% weight): Pages that historically get a lot of search impression demands 
##### Freshness Risk (30% weight): Pages that have not been updates in a long time
##### Position Opportunity (25% weight): Pages that rank on page 1 of google (position 1-50) and are within striking distance of the top spots
##### Depth Gap (5% weight): Pages with low word count

* **stale_visible_page**: High traffic but not updated in over 180 days
* **declining with demand** : High traffic but the impressions are already dropping
* **thin_visible_page** : High traffic but less the 1200 word count
* **page_one_decay_risk** : Ranks in position 1 - 10 but is older than 180 days (Risk dropping off)
* **low_ctr_visible_page** : High traffic ranks well but get very few clicks
* **low_engagement_visible_pages** : Has traffic, but engagement rate is under 30%

In [1]:
import numpy as np 
import pandas as pd 

processed_data = pd.read_csv('../../data/processed/refresh_feature_vector.csv')

processed_data.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,down,-41.4,1,8.243808,3.401197,2.890372,0.0,1,0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,down,-57.7,1,9.636980,2.079442,2.302585,0.0,1,0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,down,-60.9,1,9.440023,2.484907,2.484907,0.0,1,0,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,0.0,0.0,...,stable,-13.8,0,9.371779,4.077537,4.369448,0.0,1,0,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,down,-34.7,1,9.859588,3.218876,4.983607,0.0,1,0,1


In [14]:
# to define the fuction for reason code 
def get_reason_code(row):
    reasons = []

    if row["days_since_last_update"] > 180 and row['impressions_90d'] >= 500:
        reasons.append("stale_visible_page")
    if row['trend_direction'].lower() == "down" and row['impressions_90d'] >= 100:
        reasons.append('declining_with_demand')
    if 0 < row["word_count"] <= 1200 and row['impressions_90d'] >= 250:
        reasons.append("thin_visible_page")
    if 0 < row['avg_position'] < 10 and row['content_age_days'] >= 180:
        reasons.append('page_one_decay_risk')
    if row['ctr'] < 0.5 and 0 < row['avg_position'] < 20 and row['impressions_90d'] >= 500:
        reasons.append('low_ctr_visible_page')
    if row['sessions_90d'] >= 30 and (0 < row['engagement_rate'] < 30) or (0 < row['scroll_rate'] < 30):
        reasons.append('low_engagement_visible_pages')
        
    if not reasons: 
        reasons.append("general_refresh_review")
    return "|".join(reasons)

print("Sample pages reasons test")
print(get_reason_code(processed_data.iloc[0]))
print(get_reason_code(processed_data.iloc[4]))
print(get_reason_code(processed_data.iloc[6]))
print(get_reason_code(processed_data.iloc[50]))
print(get_reason_code(processed_data.iloc[207]))

Sample pages reasons test
declining_with_demand|low_engagement_visible_pages
declining_with_demand|low_engagement_visible_pages
general_refresh_review
declining_with_demand|low_ctr_visible_page|low_engagement_visible_pages
general_refresh_review


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
# to write a helper function to normalise the ranking
def normalize(series):
    s_min = series.min()
    s_max = series.max()

    if s_max - s_min == 0:
        return series * 0.0
    return (series - s_min) / (s_max - s_min)

def percentile_rank(series):
    return series.rank(pct=True)

# The components 
processed_data['visibility_score'] = percentile_rank(np.log1p(processed_data["impressions_90d"]))
processed_data['freshness_risk_score'] = percentile_rank(processed_data['days_since_last_update']) 

# Opportunity: ranks 1 to 50 on page one
processed_data["position_opportunity_score"] = ((1 - normalize(processed_data["avg_position"].clip(lower=1, upper=50)))
    * processed_data["visibility_score"]
    * (processed_data["avg_position"] > 0).astype(int)
)
processed_data["depth_gap_score"] = (1 - percentile_rank(processed_data["word_count"])) * processed_data["visibility_score"]

#  Computing the final baseline score
processed_data["baseline_refresh_score"] = (
    0.40 * processed_data["visibility_score"]
    + 0.30 * processed_data["freshness_risk_score"]
    + 0.25 * processed_data["position_opportunity_score"]
    + 0.05 * processed_data["depth_gap_score"]
).clip(0, 1)


In [16]:
# Applying reason codes and actions
processed_data["reason_codes"] = processed_data.apply(get_reason_code, axis=1)

In [17]:
def get_action(row):
    reasons = set(row["reason_codes"].split("|"))
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons or "declining_with_demand" in reasons:
        return "refresh"
    return "monitor"

In [18]:
processed_data["suggested_action_baseline"] = processed_data.apply(get_action, axis=1)

#  Ranking evrything
processed_data["baseline_rank"] = processed_data["baseline_refresh_score"].rank(method="first", ascending=False).astype(int)

In [20]:
# Saving the output
import os 

output_path = "../outputs/baseline_action_score.csv"


os.makedirs(os.path.dirname(output_path), exist_ok=True)
sorted_data = processed_data.sort_values("baseline_rank")
sorted_data.to_csv(output_path, index=False)

print(f"Baseline queue saved to {output_path}")

Baseline queue saved to ../outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [21]:
# Show the top 20 recommendations
top_20 = sorted_data[['baseline_rank', 'content_id', 'baseline_refresh_score','reason_codes', 'suggested_action_baseline', 'is_declining_label']].head(20)
display(top_20)


,baseline_rank,content_id,baseline_refresh_score,reason_codes,suggested_action_baseline,is_declining_label
21565,1,content_9532f197bbc8,0.941189,declining_with_demand|page_one_decay_risk|low_...,refresh,1
4644,2,content_4d1fe5b32dc2,0.934889,page_one_decay_risk|low_engagement_visible_pages,monitor,0
18954,3,content_07f2e7a6f38a,0.934080,page_one_decay_risk|low_engagement_visible_pages,monitor,0
17400,4,content_e5ae436f9a16,0.933606,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,0
9348,5,content_3430a8b94511,0.933559,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,0
25409,6,content_cbd93118300b,0.933263,declining_with_demand|page_one_decay_risk|low_...,refresh_and_review_ctr,1
18458,7,content_9c195417f6ef,0.932991,page_one_decay_risk|low_engagement_visible_pages,monitor,0
13306,8,content_ba2acb4ebd04,0.931623,page_one_decay_risk|low_engagement_visible_pages,monitor,0
28354,9,content_79b25654070a,0.931363,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,0
8275,10,content_adddad39251c,0.931124,page_one_decay_risk|low_engagement_visible_pages,monitor,0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [23]:
# The prescision @50 score. 
precision_50 = sorted_data['is_declining_label'].head(50).mean()
print(f"Baseline Precision@50: {precision_50:.3f}")
print(f"Base rate of decline in full dataset: {processed_data['is_declining_label'].mean():.3f}")


Baseline Precision@50: 0.340
Base rate of decline in full dataset: 0.542


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.